# PyTorch Tensors and Forward Propagation in Neural Networks

This notebook explains tensor basics using PyTorch and demonstrates how tensors flow through neural network layers during forward propagation.

**Reference:** https://pytorch.org/

## Table of Contents
1. Tensor Fundamentals
2. Tensor Operations
3. Connection to Neural Networks
4. Forward Propagation Implementation
5. Multi-layer Network Example

In [1]:
# Import necessary libraries
import torch
import numpy as np
import torch.nn.functional as F

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Tensor Fundamentals

### What are Tensors?
Tensors are multi-dimensional arrays with a uniform data type. They are the fundamental data structure in deep learning.

**Tensor Ranks:**
- **Scalar (Rank-0):** A single value with no dimensions (e.g., bias term in a neuron)
- **Vector (Rank-1):** 1D array with one axis (e.g., single data sample or neuron activations)
- **Matrix (Rank-2):** 2D array with two axes (e.g., weight matrix connecting two layers)
- **Higher-rank tensors:** 3D or more (e.g., batches of images, sequences)

**Connection to Neural Networks:**
- Input data: typically vectors or matrices
- Weights: matrices connecting layers
- Biases: vectors added to each layer
- Activations: outputs at each layer

In [2]:
# Rank-0 tensor (Scalar): Used for single values like learning rate or bias
rank0 = torch.tensor(1)
print("Rank-0 (Scalar) - Dimensions:", rank0.dim())
print("Value:", rank0)
print("Use case: Single bias value, learning rate\n")

# Rank-1 tensor (Vector): Used for features of a single sample or neuron outputs
rank1 = torch.tensor([1, 2, 3])
print("Rank-1 (Vector) - Dimensions:", rank1.dim())
print("Value:", rank1)
print("Shape:", rank1.shape)
print("Use case: Single input sample with 3 features\n")

# Rank-2 tensor (Matrix): Used for weights connecting two layers or batch of samples
rank2 = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("Rank-2 (Matrix) - Dimensions:", rank2.dim())
print("Value:", rank2)
print("Shape:", rank2.shape, "(2 samples, 3 features each)")
print("Use case: Weight matrix or batch of 2 input samples\n")

# Rank-3 tensor: Used for sequences or batches of images
rank3 = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print("Rank-3 - Dimensions:", rank3.dim())
print("Value:", rank3)
print("Shape:", rank3.shape, "(batch_size, height, width)")
print("Use case: Batch of 2 grayscale images of 2x2 pixels")

Rank-0 (Scalar) - Dimensions: 0
Value: tensor(1)
Use case: Single bias value, learning rate

Rank-1 (Vector) - Dimensions: 1
Value: tensor([1, 2, 3])
Shape: torch.Size([3])
Use case: Single input sample with 3 features

Rank-2 (Matrix) - Dimensions: 2
Value: tensor([[1, 2, 3],
        [4, 5, 6]])
Shape: torch.Size([2, 3]) (2 samples, 3 features each)
Use case: Weight matrix or batch of 2 input samples

Rank-3 - Dimensions: 3
Value: tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
Shape: torch.Size([2, 2, 2]) (batch_size, height, width)
Use case: Batch of 2 grayscale images of 2x2 pixels


### Tensor Data Types

- **float32 (default):** Standard for weights and activations
- **float16:** Used for mixed-precision training to save memory
- **int64:** Used for labels and indices
- **bool:** Used for masks and attention mechanisms

In [3]:
# Integer tensor (default for whole numbers)
# Use case: Class labels in classification
tensor1d = torch.tensor([1, 2, 3])
print("Integer tensor dtype:", tensor1d.dtype)

# Float tensor (required for neural network parameters)
# PyTorch creates 32-bit float tensors from Python floats
floatvec = torch.tensor([1.0, 2.0, 3.0])
print("Float tensor dtype:", floatvec.dtype)

# Converting data types
# Important: Always use float32 for weights and activations
floatvec1 = tensor1d.to(torch.float32)
print("Converted to float32:", floatvec1.dtype)
print("Converted values:", floatvec1)

Integer tensor dtype: torch.int64
Float tensor dtype: torch.float32
Converted to float32: torch.float32
Converted values: tensor([1., 2., 3.])


### Initializing Tensors

- **Random initialization:** Breaks symmetry in weight matrices
- **Zeros:** Sometimes used for biases
- **Ones:** Rarely used directly, but useful for masks
- **Xavier/He initialization:** Specialized methods for better training

In [4]:
# Method 1: From Python list
# Use case: Manually defining small weight matrices or test data
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
print("Tensor from list:\n", x_data)

# Method 2: From NumPy array
# Use case: Converting preprocessed data from NumPy
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print("\nTensor from NumPy:\n", x_np)

Tensor from list:
 tensor([[1, 2],
        [3, 4]])

Tensor from NumPy:
 tensor([[1, 2],
        [3, 4]])


In [5]:
# Creating tensors with specific initialization patterns
# These are commonly used in neural network initialization

shape = (2, 3)  # 2 neurons, 3 input features

# Random initialization (uniform distribution [0, 1))
# Use case: Initial weights for neural network layers
rand_tensor = torch.rand(shape)
print(f"Random Tensor (uniform [0,1)):\n {rand_tensor}\n")

# Normal distribution initialization (mean=0, std=1)
# Use case: Xavier/He initialization for deeper networks
randn_tensor = torch.randn(shape)
print(f"Random Tensor (normal distribution):\n {randn_tensor}\n")

# Ones tensor
# Use case: Creating masks or initial attention weights
ones_tensor = torch.ones(shape)
print(f"Ones Tensor:\n {ones_tensor}\n")

# Zeros tensor
# Use case: Initial bias values (common practice)
zeros_tensor = torch.zeros(shape)
print(f"Zeros Tensor (typical for biases):\n {zeros_tensor}\n")

# Tensor properties important for debugging
print(f"Shape of tensor: {rand_tensor.shape}")
print(f"Data type: {rand_tensor.dtype}")
print(f"Device (CPU/GPU): {rand_tensor.device}")

Random Tensor (uniform [0,1)):
 tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])

Random Tensor (normal distribution):
 tensor([[ 1.1561,  0.3965, -2.4661],
        [ 0.3623,  0.3765, -0.1808]])

Ones Tensor:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

Zeros Tensor (typical for biases):
 tensor([[0., 0., 0.],
        [0., 0., 0.]])

Shape of tensor: torch.Size([2, 3])
Data type: torch.float32
Device (CPU/GPU): cpu


## 2. Tensor Operations for Neural Networks

Understanding these operations is essential for implementing forward propagation.

**Key operations:**
- **Matrix multiplication:** Core of linear layers (y = Wx + b)
- **Element-wise operations:** Activation functions
- **Concatenation/Stacking:** Combining features or batches
- **Indexing/Slicing:** Selecting specific neurons or samples

In [6]:
# GPU acceleration for neural networks
# Moving tensors to GPU significantly speeds up training

if torch.cuda.is_available():
    tensor = rand_tensor.to("cuda")
    print("Tensor moved to GPU:", tensor.device)
else:
    print("GPU not available, using CPU")
    print("For large models, GPU is essential for reasonable training time")

GPU not available, using CPU
For large models, GPU is essential for reasonable training time


In [7]:
# Indexing and slicing operations
# Use case: Selecting specific samples, features, or neurons

tensor = torch.rand(3, 4)  # 3 samples, 4 features each
print("Original tensor:\n", tensor)

# Access first sample (all features)
print(f"\nFirst sample (row): {tensor[0]}")
print("Use case: Extracting a single data point from batch")

# Access first feature (across all samples)
print(f"\nFirst feature (column): {tensor[:, 0]}")
print("Use case: Extracting one feature dimension")

# Access last feature
print(f"\nLast feature: {tensor[..., -1]}")

# Modify specific feature (simulating dropout or feature masking)
tensor[:, 1] = 0  # Zero out second feature
print("\nAfter masking second feature:\n", tensor)
print("Use case: Dropout or feature selection")

Original tensor:
 tensor([[0.2666, 0.6274, 0.2696, 0.4414],
        [0.2969, 0.8317, 0.1053, 0.2695],
        [0.3588, 0.1994, 0.5472, 0.0062]])

First sample (row): tensor([0.2666, 0.6274, 0.2696, 0.4414])
Use case: Extracting a single data point from batch

First feature (column): tensor([0.2666, 0.2969, 0.3588])
Use case: Extracting one feature dimension

Last feature: tensor([0.4414, 0.2695, 0.0062])

After masking second feature:
 tensor([[0.2666, 0.0000, 0.2696, 0.4414],
        [0.2969, 0.0000, 0.1053, 0.2695],
        [0.3588, 0.0000, 0.5472, 0.0062]])
Use case: Dropout or feature selection


In [10]:
# Concatenation: Combining tensors along existing dimensions
# Use case: Merging features from different sources or batches

tensor = torch.ones(3, 4)

# Concatenate along columns (adding more features)
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print("Original shape:", tensor.shape)
print("After concatenation (dim=1):", t1.shape)
print(t1)
print("Use case: Combining features from multiple layers (skip connections)\n")

# Concatenate along rows (adding more samples)
t2 = torch.cat([tensor, tensor], dim=0)
print("After concatenation (dim=0):", t2.shape)
print(t2)
print("Use case: Combining multiple batches of data")

Original shape: torch.Size([3, 4])
After concatenation (dim=1): torch.Size([3, 12])
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
Use case: Combining features from multiple layers (skip connections)

After concatenation (dim=0): torch.Size([6, 4])
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
Use case: Combining multiple batches of data


In [9]:
# Stacking: Creating new dimension
# Use case: Creating batches from individual samples

# Individual samples (vectors representing different inputs)
sample1 = torch.tensor([1, 2, 3])  # Sample 1: 3 features
sample2 = torch.tensor([4, 5, 6])  # Sample 2: 3 features
sample3 = torch.tensor([7, 8, 9])  # Sample 3: 3 features
sample4 = torch.tensor([10, 11, 12])  # Sample 4: 3 features

print("Single sample shape:", sample1.shape)
print("Single sample dimensions:", sample1.dim())
print("Single sample tensor:\n", sample1)

# Stack along dimension 0: Creates batch dimension
# Result: (batch_size, features)
batch = torch.stack((sample1, sample2, sample3, sample4), dim=0)
print("\nBatch shape after stacking (dim=0):", batch.shape)
print("Batch dimensions:", batch.dim())
print("Batch tensor:\n", batch)
print("Interpretation: 4 samples, 3 features each")

# Stack along dimension 1: Less common, transposes the relationship
features_first = torch.stack((sample1, sample2, sample3, sample4), dim=1)
print("\nShape after stacking (dim=1):", features_first.shape)
print("Tensor:\n", features_first)
print("Interpretation: 3 timesteps, 4 features at each timestep")

Single sample shape: torch.Size([3])
Single sample dimensions: 1
Single sample tensor:
 tensor([1, 2, 3])

Batch shape after stacking (dim=0): torch.Size([4, 3])
Batch dimensions: 2
Batch tensor:
 tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]])
Interpretation: 4 samples, 3 features each

Shape after stacking (dim=1): torch.Size([3, 4])
Tensor:
 tensor([[ 1,  4,  7, 10],
        [ 2,  5,  8, 11],
        [ 3,  6,  9, 12]])
Interpretation: 3 timesteps, 4 features at each timestep


### Matrix Multiplication - Core of Neural Networks

**Matrix multiplication is the fundamental operation in forward propagation:**

Linear transformation: $y = Wx + b$

Where:
- $x$: Input features (shape: batch_size × input_features)
- $W$: Weight matrix (shape: input_features × output_features)
- $b$: Bias vector (shape: output_features)
- $y$: Output activations (shape: batch_size × output_features)

**Important:** Matrix dimensions must align: (m×n) @ (n×p) = (m×p)

In [11]:
# Matrix multiplication examples
# These operations represent neurons computing weighted sums

# Create a simple tensor
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print("Matrix A shape:", A.shape, "(2 samples, 3 features)")
print("Matrix A:\n", A)

# Transpose
print("\nA.T shape:", A.T.shape, "(3 features, 2 samples)")
print("A.T:\n", A.T)

# Method 1: @ operator (preferred for readability)
result1 = A @ A.T
print("\nA @ A.T (using @ operator):\n", result1)
print("Result shape:", result1.shape)

# Method 2: matmul function
result2 = torch.matmul(A, A.T)
print("\nA @ A.T (using matmul):\n", result2)

# Method 3: mm (only for 2D matrices)
result3 = A.mm(A.T)

print("\nA @ A.T (using mm):\n", result3)

# All methods produce the same result
print("\nAll methods equal:", torch.equal(result1, result2) and torch.equal(result2, result3))

Matrix A shape: torch.Size([2, 3]) (2 samples, 3 features)
Matrix A:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])

A.T shape: torch.Size([3, 2]) (3 features, 2 samples)
A.T:
 tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])

A @ A.T (using @ operator):
 tensor([[14., 32.],
        [32., 77.]])
Result shape: torch.Size([2, 2])

A @ A.T (using matmul):
 tensor([[14., 32.],
        [32., 77.]])

A @ A.T (using mm):
 tensor([[14., 32.],
        [32., 77.]])

All methods equal: True


## 3. Connection to Neural Networks

### How Tensors Represent Neural Network Components

**A single neuron computation:**
1. Takes input vector: $x = [x_1, x_2, ..., x_n]$
2. Multiplies by weights: $w = [w_1, w_2, ..., w_n]$
3. Adds bias: $b$
4. Applies activation: $a = \sigma(w^T x + b)$

**A layer of neurons:**
- Weight matrix $W$ where each row represents one neuron's weights
- Bias vector $b$ with one element per neuron
- Output: $y = \sigma(Wx + b)$

In [12]:
# Single neuron computation
# This is the building block of neural networks

print("SINGLE NEURON COMPUTATION ")

# Input: single sample with 3 features
x = torch.tensor([1.0, 2.0, 3.0])
print("Input x:", x)
print("Shape:", x.shape, "(3 features)\n")

# Weights: one weight per input feature
w = torch.tensor([0.5, -0.3, 0.8])
print("Weights w:", w)
print("Shape:", w.shape, "(3 weights, one per input)\n")

# Bias: single value for this neuron
b = torch.tensor(0.1)
print("Bias b:", b.item(), "\n")

# Linear combination (weighted sum)
z = torch.dot(x, w) + b
print("Linear output z = w·x + b =", z.item())
print("Calculation: ({:.1f}×{:.1f}) + ({:.1f}×{:.1f}) + ({:.1f}×{:.1f}) + {:.1f} = {:.2f}\n".format(
    x[0].item(), w[0].item(),
    x[1].item(), w[1].item(),
    x[2].item(), w[2].item(),
    b.item(), z.item()))

# Apply activation function (ReLU)
# ReLU(z) = max(0, z)
activation = F.relu(z)
print("After ReLU activation:", activation.item())
print("This is the output of our single neuron!")

SINGLE NEURON COMPUTATION 
Input x: tensor([1., 2., 3.])
Shape: torch.Size([3]) (3 features)

Weights w: tensor([ 0.5000, -0.3000,  0.8000])
Shape: torch.Size([3]) (3 weights, one per input)

Bias b: 0.10000000149011612 

Linear output z = w·x + b = 2.4000000953674316
Calculation: (1.0×0.5) + (2.0×-0.3) + (3.0×0.8) + 0.1 = 2.40

After ReLU activation: 2.4000000953674316
This is the output of our single neuron!


In [ ]:
# Layer of neurons computation
# Multiple neurons processing the same input in parallel

print("LAYER OF NEURONS COMPUTATION ")

# Input: single sample with 3 features
x = torch.tensor([1.0, 2.0, 3.0])
print("Input x:", x)
print("Shape:", x.shape, "(3 features)\n")

# Weight matrix: 4 neurons, each with 3 weights i.e input layer 3 nodes, hidden layer 4 neurons
# Each row represents one neuron's weights
W = torch.tensor([[0.5, -0.3, 0.8],   # Neuron 1 weights
                  [0.2, 0.9, -0.5],   # Neuron 2 weights
                  [-0.7, 0.4, 0.6],   # Neuron 3 weights
                  [0.3, -0.8, 0.1]])  # Neuron 4 weights
print("Weight matrix W:\n", W)
print("Shape:", W.shape, "(4 neurons, 3 inputs each)\n")

# Bias vector: one bias per neuron
b = torch.tensor([0.1, -0.2, 0.3, 0.0])
print("Bias vector b:", b)
print("Shape:", b.shape, "(4 biases, one per neuron)\n")

# Linear transformation: matrix-vector multiplication
# This computes all neurons in parallel!
z = W @ x + b  # Equivalent to: W.matmul(x) + b
print("Linear outputs z = Wx + b:", z)
print("Shape:", z.shape, "(4 neurons)\n")

# Apply activation function element-wise
a = F.relu(z)
print("After ReLU activation:", a)
print("This is the output of our layer with 4 neurons!")
print("\nNote: Negative values became 0 due to ReLU")

LAYER OF NEURONS COMPUTATION 
Input x: tensor([1., 2., 3.])
Shape: torch.Size([3]) (3 features)

Weight matrix W:
 tensor([[ 0.5000, -0.3000,  0.8000],
        [ 0.2000,  0.9000, -0.5000],
        [-0.7000,  0.4000,  0.6000],
        [ 0.3000, -0.8000,  0.1000]])
Shape: torch.Size([4, 3]) (4 neurons, 3 inputs each)

Bias vector b: tensor([ 0.1000, -0.2000,  0.3000,  0.0000])
Shape: torch.Size([4]) (4 biases, one per neuron)

Linear outputs z = Wx + b: tensor([ 2.4000,  0.3000,  2.2000, -1.0000])
Shape: torch.Size([4]) (4 neurons)

After ReLU activation: tensor([2.4000, 0.3000, 2.2000, 0.0000])
This is the output of our layer with 4 neurons!

Note: Negative values became 0 due to ReLU


## 4. Forward Propagation Implementation

**Forward propagation** is the process of passing input through the network layer by layer.

**Process for each layer:**
1. Linear transformation: $z = Wx + b$
2. Non-linear activation: $a = \sigma(z)$
3. Pass output to next layer

**Common activation functions:**
- **ReLU:** $\text{ReLU}(x) = \max(0, x)$ (most common in hidden layers)
- **Sigmoid:** $\sigma(x) = \frac{1}{1 + e^{-x}}$ (binary classification output)
- **Softmax:** $\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$ (multi-class classification)
- **Tanh:** $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ (sometimes in RNNs)

In [ ]:
# Comparing different activation functions
# Understanding activations is crucial for network design

# Sample pre-activation values
z = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Pre-activation values z:", z, "\n")

# ReLU: Most popular for hidden layers
relu_output = F.relu(z)
print("ReLU(z):", relu_output)
print("Property: Outputs 0 for negative inputs, identity for positive\n")

# Sigmoid: Squashes values to (0, 1)
sigmoid_output = torch.sigmoid(z)
print("Sigmoid(z):", sigmoid_output)
print("Property: Output range (0, 1), used for probabilities\n")

# Tanh: Squashes values to (-1, 1)
tanh_output = torch.tanh(z)
print("Tanh(z):", tanh_output)
print("Property: Output range (-1, 1), zero-centered\n")

# Softmax: Converts to probability distribution
softmax_output = F.softmax(z, dim=0)
print("Softmax(z):", softmax_output)
print("Sum of softmax outputs:", softmax_output.sum().item())
print("Property: Outputs sum to 1, used for multi-class classification")

Pre-activation values z: tensor([-2., -1.,  0.,  1.,  2.]) 

ReLU(z): tensor([0., 0., 0., 1., 2.])
Property: Outputs 0 for negative inputs, identity for positive

Sigmoid(z): tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
Property: Output range (0, 1), used for probabilities

Tanh(z): tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])
Property: Output range (-1, 1), zero-centered

Softmax(z): tensor([0.0117, 0.0317, 0.0861, 0.2341, 0.6364])
Sum of softmax outputs: 1.0
Property: Outputs sum to 1, used for multi-class classification


In [ ]:
# Forward propagation with batches
# Real networks process multiple samples simultaneously

print("BATCH FORWARD PROPAGATION ")

# Batch of inputs: 3 samples, each with 5 features
X_batch = torch.randn(3, 5)
print("Input batch X shape:", X_batch.shape, "(3 samples, 5 features)")
print("Input batch:\n", X_batch, "\n")

# Layer 1: 5 inputs -> 4 neurons
W1 = torch.randn(5, 4) * 0.5  # Initialize with small random values
b1 = torch.zeros(4)  # Initialize biases to zero
print("Layer 1 weights W1 shape:", W1.shape, "(5 inputs, 4 neurons)")
print("Layer 1 biases b1 shape:", b1.shape, "\n")

# Forward pass through layer 1
# Note: X @ W broadcasts automatically for batches
Z1 = X_batch @ W1 + b1  # Linear transformation
A1 = F.relu(Z1)  # Activation
print("Layer 1 output shape:", A1.shape, "(3 samples, 4 neurons)")
print("Layer 1 activations:\n", A1, "\n")

# Layer 2: 4 inputs -> 3 neurons (output layer)
W2 = torch.randn(4, 3) * 0.5
b2 = torch.zeros(3)
print("Layer 2 weights W2 shape:", W2.shape, "(4 inputs, 3 neurons)")
print("Layer 2 biases b2 shape:", b2.shape, "\n")

# Forward pass through layer 2 (output layer)
Z2 = A1 @ W2 + b2  # Linear transformation
A2 = F.softmax(Z2, dim=1)  # Softmax for classification
print("Final output shape:", A2.shape, "(3 samples, 3 classes)")
print("Class probabilities for each sample:\n", A2)
print("\nEach row sums to 1:", A2.sum(dim=1))

BATCH FORWARD PROPAGATION 
Input batch X shape: torch.Size([3, 5]) (3 samples, 5 features)
Input batch:
 tensor([[-0.3267, -0.2788, -0.4220, -1.3323, -0.3639],
        [ 0.1513, -0.3514, -0.7906, -0.0915,  0.2352],
        [ 2.2440,  0.5817,  0.4528,  0.6410,  0.5200]]) 

Layer 1 weights W1 shape: torch.Size([5, 4]) (5 inputs, 4 neurons)
Layer 1 biases b1 shape: torch.Size([4]) 

Layer 1 output shape: torch.Size([3, 4]) (3 samples, 4 neurons)
Layer 1 activations:
 tensor([[0.1761, 0.0000, 0.7188, 0.2575],
        [0.0942, 0.0779, 0.0000, 0.4406],
        [1.4332, 0.0000, 1.3388, 1.0170]]) 

Layer 2 weights W2 shape: torch.Size([4, 3]) (4 inputs, 3 neurons)
Layer 2 biases b2 shape: torch.Size([3]) 

Final output shape: torch.Size([3, 3]) (3 samples, 3 classes)
Class probabilities for each sample:
 tensor([[0.2414, 0.4440, 0.3146],
        [0.2794, 0.4029, 0.3177],
        [0.2602, 0.4758, 0.2640]])

Each row sums to 1: tensor([1.0000, 1.0000, 1.0000])


## 5. Complete Multi-layer Network Example

**Network architecture:**
- Input layer: 4 features
- Hidden layer 1: 8 neurons (ReLU)
- Hidden layer 2: 6 neurons (ReLU)
- Output layer: 3 neurons (Softmax for 3-class classification)

In [ ]:
# Complete implementation of forward propagation
# This is a simplified version of what PyTorch nn.Module does

def forward_propagation(X, W1, b1, W2, b2, W3, b3):
    """
    Forward propagation through a 3-layer network.

    Args:
        X: Input data (batch_size, input_features)
        W1, b1: Layer 1 parameters
        W2, b2: Layer 2 parameters
        W3, b3: Layer 3 parameters (output)

    Returns:
        output: Network predictions
        cache: Dictionary of intermediate values (for backprop)
    """
    # Layer 1: Input -> Hidden1
    Z1 = X @ W1 + b1  # Linear transformation
    A1 = F.relu(Z1)    # ReLU activation

    # Layer 2: Hidden1 -> Hidden2
    Z2 = A1 @ W2 + b2  # Linear transformation
    A2 = F.relu(Z2)     # ReLU activation

    # Layer 3: Hidden2 -> Output
    Z3 = A2 @ W3 + b3         # Linear transformation
    A3 = F.softmax(Z3, dim=1)  # Softmax activation

    # Cache intermediate values (needed for backpropagation)
    cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2, 'Z3': Z3, 'A3': A3}

    return A3, cache

# Network configuration
input_size = 4 # 4 features
hidden1_size = 8 # 8 neurons
hidden2_size = 6
output_size = 3 # 3 classes
batch_size = 5 # 5 examples

# Initialize parameters with proper scaling
# Xavier initialization: scale by sqrt(1/n_inputs)
W1 = torch.randn(input_size, hidden1_size) * np.sqrt(1/input_size)
b1 = torch.zeros(hidden1_size)

W2 = torch.randn(hidden1_size, hidden2_size) * np.sqrt(1/hidden1_size)
b2 = torch.zeros(hidden2_size)

W3 = torch.randn(hidden2_size, output_size) * np.sqrt(1/hidden2_size)
b3 = torch.zeros(output_size)

# Generate sample input data
X = torch.randn(batch_size, input_size)

print(" COMPLETE NEURAL NETWORK FORWARD PROPAGATION ")
print(f"\nNetwork Architecture:")
print(f"  Input Layer: {input_size} features")
print(f"  Hidden Layer 1: {hidden1_size} neurons (ReLU)")
print(f"  Hidden Layer 2: {hidden2_size} neurons (ReLU)")
print(f"  Output Layer: {output_size} neurons (Softmax)")
print(f"\nProcessing batch of {batch_size} samples...\n")

# Perform forward propagation
predictions, cache = forward_propagation(X, W1, b1, W2, b2, W3, b3)

# Display results
print("Input shape:", X.shape)
print("Hidden layer 1 output shape:", cache['A1'].shape)
print("Hidden layer 2 output shape:", cache['A2'].shape)
print("Final output shape:", predictions.shape)

print("\nClass probabilities for each sample:")
print(predictions)

# Get predicted classes (highest probability)
predicted_classes = torch.argmax(predictions, dim=1)
print("\nPredicted classes:", predicted_classes)

# Verify probabilities sum to 1
print("\nVerification - each row sums to 1.0:")
print(predictions.sum(dim=1))

 COMPLETE NEURAL NETWORK FORWARD PROPAGATION 

Network Architecture:
  Input Layer: 4 features
  Hidden Layer 1: 8 neurons (ReLU)
  Hidden Layer 2: 6 neurons (ReLU)
  Output Layer: 3 neurons (Softmax)

Processing batch of 5 samples...

Input shape: torch.Size([5, 4])
Hidden layer 1 output shape: torch.Size([5, 8])
Hidden layer 2 output shape: torch.Size([5, 6])
Final output shape: torch.Size([5, 3])

Class probabilities for each sample:
tensor([[0.3394, 0.3237, 0.3369],
        [0.4220, 0.2367, 0.3413],
        [0.3333, 0.3333, 0.3333],
        [0.3340, 0.3256, 0.3404],
        [0.3280, 0.3596, 0.3125]])

Predicted classes: tensor([0, 0, 0, 2, 1])

Verification - each row sums to 1.0:
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


### Tracking Tensor Shapes Through the Network

**Shape transformations are critical to understand:**

| Layer | Operation | Input Shape | Weight Shape | Output Shape |
|-------|-----------|-------------|--------------|---------------|
| Input | - | (batch, 4) | - | (batch, 4) |
| Hidden 1 | Linear + ReLU | (batch, 4) | (4, 8) | (batch, 8) |
| Hidden 2 | Linear + ReLU | (batch, 8) | (8, 6) | (batch, 6) |
| Output | Linear + Softmax | (batch, 6) | (6, 3) | (batch, 3) |

**Key principle:** For matrix multiplication A @ B, inner dimensions must match!
- (m × n) @ (n × p) = (m × p)
- (m × n) @ (k × p) = ERROR if n ≠ k

In [ ]:
# Debugging shape mismatches - common errors in neural networks

print(" COMMON SHAPE ERRORS AND HOW TO FIX THEM ")

# Example 1: Correct shape alignment
x = torch.randn(3, 4)  # 3 samples, 4 features
w = torch.randn(4, 5)  # 4 inputs, 5 outputs
print("\n1. CORRECT:")
print(f"   x shape: {x.shape}, w shape: {w.shape}")
result = x @ w
print(f"   Result shape: {result.shape} ")

# Example 2: Shape mismatch
print("\n2. ERROR - Shape mismatch:")
x = torch.randn(3, 4)
w_wrong = torch.randn(5, 6)  # Wrong! 4 ≠ 5
print(f"   x shape: {x.shape}, w shape: {w_wrong.shape}")
try:
    result = x @ w_wrong
except RuntimeError as e:
    print(f"   Error: {e} ")

# Example 3: Fixing with transpose
print("\n3. FIX - Use transpose when needed:")
x = torch.randn(3, 4)
w = torch.randn(5, 4)  # 5 outputs, 4 inputs
print(f"   x shape: {x.shape}, w shape: {w.shape}")
print(f"   Use w.T: {w.T.shape}")
result = x @ w.T
print(f"   Result shape: {result.shape} ")

# Example 4: Adding bias - broadcasting
print("\n4. BROADCASTING - Adding bias:")
x = torch.randn(3, 4)
b = torch.randn(4)
print(f"   x shape: {x.shape}, b shape: {b.shape}")
result = x + b  # b is automatically broadcast to (3, 4)
print(f"   Result shape: {result.shape} ")
print("   Each row of x gets the same bias added")

 COMMON SHAPE ERRORS AND HOW TO FIX THEM 

1. CORRECT:
   x shape: torch.Size([3, 4]), w shape: torch.Size([4, 5])
   Result shape: torch.Size([3, 5]) 

2. ERROR - Shape mismatch:
   x shape: torch.Size([3, 4]), w shape: torch.Size([5, 6])
   Error: mat1 and mat2 shapes cannot be multiplied (3x4 and 5x6) 

3. FIX - Use transpose when needed:
   x shape: torch.Size([3, 4]), w shape: torch.Size([5, 4])
   Use w.T: torch.Size([4, 5])
   Result shape: torch.Size([3, 5]) 

4. BROADCASTING - Adding bias:
   x shape: torch.Size([3, 4]), b shape: torch.Size([4])
   Result shape: torch.Size([3, 4]) 
   Each row of x gets the same bias added


## Key Takeaways for Neural Network Implementation

### 1. Tensor Dimensions
- **Always** include batch dimension as first dimension
- Shape format: (batch_size, features) for 2D data
- Shape format: (batch_size, sequence_length, features) for sequences

### 2. Weight Matrices
- Shape: (input_features, output_features)
- Each column represents one output neuron's weights
- Initialize carefully (Xavier, He initialization)

### 3. Forward Propagation Pattern
```python
# For each layer:
Z = X @ W + b      # Linear transformation
A = activation(Z)   # Non-linear activation
X = A               # Output becomes input to next layer
```

### 4. Activation Functions
- **Hidden layers:** Usually ReLU or variants (LeakyReLU, ELU)
- **Binary classification output:** Sigmoid
- **Multi-class classification output:** Softmax
- **Regression output:** No activation (linear)

### 5. Common Debugging Steps
1. Print shapes after each operation
2. Check for NaN or Inf values
3. Verify matrix multiplication dimensions align
4. Ensure activations are in expected range
5. Confirm batch dimension is preserved

In [ ]:
# Comparison: Manual implementation vs PyTorch nn.Module
# This shows how our manual implementation relates to standard PyTorch

import torch.nn as nn

print("  MANUAL vs PYTORCH IMPLEMENTATION ")

# Define network architecture
input_size, hidden_size, output_size = 4, 8, 3
batch_size = 2

# Sample input
X = torch.randn(batch_size, input_size)

# Method 1: Manual implementation
print("\n1. MANUAL IMPLEMENTATION:")
W1 = torch.randn(input_size, hidden_size) * 0.1
b1 = torch.zeros(hidden_size)
W2 = torch.randn(hidden_size, output_size) * 0.1
b2 = torch.zeros(output_size)

Z1 = X @ W1 + b1
A1 = F.relu(Z1)
Z2 = A1 @ W2 + b2
output_manual = F.softmax(Z2, dim=1)
print("Output shape:", output_manual.shape)
print("Output:\n", output_manual)

# Method 2: PyTorch nn.Module (standard way)
print("\n2. PYTORCH nn.Module:")
model = nn.Sequential(
    nn.Linear(input_size, hidden_size),  # Combines W@x + b
    nn.ReLU(),                           # Activation
    nn.Linear(hidden_size, output_size), # Second layer
    nn.Softmax(dim=1)                    # Output activation
)

output_pytorch = model(X)
print("Output shape:", output_pytorch.shape)
print("Output:\n", output_pytorch)

print("\n3. WHAT PYTORCH PROVIDES:")
print("   - Automatic parameter management")
print("   - Automatic gradient computation (autograd)")
print("   - Optimized implementations")
print("   - Easy model saving/loading")
print("   - GPU acceleration")
print("\nBut understanding the manual implementation helps you:")
print("   - Debug shape errors")
print("   - Implement custom layers")
print("   - Understand what's happening inside")

  MANUAL vs PYTORCH IMPLEMENTATION 

1. MANUAL IMPLEMENTATION:
Output shape: torch.Size([2, 3])
Output:
 tensor([[0.3330, 0.3188, 0.3482],
        [0.3332, 0.3261, 0.3407]])

2. PYTORCH nn.Module:
Output shape: torch.Size([2, 3])
Output:
 tensor([[0.3950, 0.2475, 0.3575],
        [0.3992, 0.2711, 0.3297]], grad_fn=<SoftmaxBackward0>)

3. WHAT PYTORCH PROVIDES:
   - Automatic parameter management
   - Automatic gradient computation (autograd)
   - Optimized implementations
   - Easy model saving/loading
   - GPU acceleration

But understanding the manual implementation helps you:
   - Debug shape errors
   - Implement custom layers
   - Understand what's happening inside


## Practice Exercises

### Exercise 1: Shape Prediction
Predict the output shape without running the code:
```python
X = torch.randn(10, 5)   # 10 samples, 5 features
W = torch.randn(5, 3)    # Layer weights
b = torch.randn(3)       # Biases
output = F.relu(X @ W + b)
# What is output.shape?
```

### Exercise 2: Multi-layer Network
Implement a 4-layer network:
- Input: 10 features
- Hidden 1: 20 neurons (ReLU)
- Hidden 2: 15 neurons (ReLU)
- Hidden 3: 10 neurons (ReLU)
- Output: 5 classes (Softmax)

### Exercise 3: Batch Processing
Modify your network to handle variable batch sizes.
Test with batch_size = 1, 16, and 128.

### Exercise 4: Activation Comparison
Compare ReLU, Sigmoid, and Tanh activations:
- Which activations can cause vanishing gradients?
- When would you use each one?
- Create plots showing their behavior